# Migration Validation Metrics (DPC1.0 → DPC2.0)

Collects metadata metrics for the Hive tables on both platforms and compares them, using the
same checks the NX1 MapR/HDFS → S3 migration DAG runs after each copy. It works no matter how the
data was copied. Nothing is changed on either platform: the collector only runs `SHOW`,
`DESCRIBE`, `SELECT COUNT(*)` and storage listings.

### Where each side is collected

- **DPC1.0 (source)** has no JupyterHub. The notebook connects to the DPC1.0 **edge node over
  SSH**, uploads `migration_metrics_collector.py` and runs it with `spark-submit`, the same way the
  migration DAG runs its discovery there. It then downloads the result.
- **DPC2.0 (destination)** is collected in this JupyterHub session's Spark.

Both sides are measured by the same `migration_metrics_collector.py`, so any difference in the
report comes from the data, not from the tooling. **Upload it to the same JupyterHub folder as
this notebook.**

### What is checked per table

| Metric | How it is collected | Mismatch → |
|---|---|---|
| Row count (total and per partition) | `COUNT(*)` grouped by the partition columns | **FAIL** |
| Schema: non-partition columns, name + type | `spark.table(...).schema` | **FAIL** |
| Partition columns, name + type | `DESCRIBE TABLE EXTENDED` + schema | **FAIL** |
| Registered partitions | `SHOW PARTITIONS` | WARN |
| Data files and bytes (total and per partition) | listing under the table `LOCATION`, skipping hidden files (`_SUCCESS`, `.crc`, `_temporary`, …) | WARN |
| Partition directories that hold data but are not registered | storage listing vs `SHOW PARTITIONS` | WARN (fix with `MSCK REPAIR TABLE`) |

Other statuses: `MISSING_ON_DEST` (table not found on DPC2.0), `EXTRA_ON_DEST` (table only on
DPC2.0), `ERROR` (metrics could not be collected), `SKIPPED_VIEW` (views hold no data).

### How to run

1. Fill in the configuration cell. The edge node SSH settings come from environment variables or
   a password prompt, so no credentials are stored in the notebook.
2. Make sure the SSH user has a valid Kerberos (or MapR) ticket on the edge node, as for the
   migration DAG. The notebook checks this before running anything.
3. Run all cells.

Each side's metrics are saved as `source_metrics.json` / `dest_metrics.json` under
`OUTPUT_DIR/RUN_LABEL`, and the comparison only reads those files. To compare again without
re-collecting (e.g. to regenerate the report), set `SIDES_TO_COLLECT = []`. To re-collect only
one side, list just that side.

### Outputs

```
OUTPUT_DIR/RUN_LABEL/
├── source_metrics.json             # raw metrics, one record per table
├── dest_metrics.json
└── report_<UTC timestamp>/
    ├── summary.csv                 # one row per table, with status and every metric
    ├── partition_diffs.csv         # every partition that differs
    └── report.html
```

### Limits

- Storage metrics only cover files under the table `LOCATION`. Row counts and `SHOW PARTITIONS` still
  include partitions that live elsewhere.
- On Iceberg tables, the storage listing also counts files left behind by expired snapshots that
  have not been cleaned up yet, and `SHOW PARTITIONS` is skipped.
- Metrics are a point-in-time snapshot. If tables are still being written to, collect both sides
  again close together.
- `COUNT(*)` on Parquet/ORC is answered from file footers. Text tables are scanned in full.

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────────────
import os

# Groups the snapshots and reports of one validation exercise under OUTPUT_DIR/RUN_LABEL.
RUN_LABEL = "dpc1-to-dpc2"

# Where snapshots and reports are written: an s3a:// path or a local directory.
OUTPUT_DIR = os.environ.get("MIGRATION_VALIDATION_DIR", "s3a://CHANGE_ME/migration-validation")

SIDES_TO_COLLECT = ["source", "dest"]

# collect_on: "edge" runs the collector on the EDGE node below over SSH; "local" uses this
# notebook's Spark session. "catalog" is prefixed to every table name; empty = default catalog.
SOURCE = {"label": "DPC1.0", "catalog": "", "collect_on": "edge"}
DEST = {"label": "DPC2.0", "catalog": "", "collect_on": "local"}

# tables / exclude: glob patterns matched against table names (case-insensitive).
# partition_filter: optional Spark SQL predicate on partition columns, applied to both sides.
# Use it when only part of a table was migrated, e.g. "dt >= '2024-01-01'".
DATABASE_PAIRS = [
    {"source_db": "CHANGE_ME", "dest_db": "CHANGE_ME", "tables": ["*"], "exclude": [], "partition_filter": ""},
]

EDGE = {
    "host": os.environ.get("EDGE_SSH_HOST", ""),
    "port": int(os.environ.get("EDGE_SSH_PORT", "22")),
    "user": os.environ.get("EDGE_SSH_USER", ""),
    # Private key file; without one, EDGE_SSH_PASSWORD is used, or you are prompted.
    "key_file": os.environ.get("EDGE_SSH_KEY_FILE", ""),
    # As in the migration DAG: "MapR" sources ~/.profile, anything else runs in a bash login shell.
    "cluster_type": "CDP",
    # "kinit" or "mapr": a valid ticket for the SSH user must already exist. "none" skips the check.
    "auth": "kinit",
    # Remote folder for the collector, its config, output and logs; expanded by the edge node's shell.
    "work_dir": "/tmp/migration_validation_$(id -un)",
    # Use spark3-submit if that is how Spark 3 is installed; add e.g. --driver-memory 8g for big databases.
    "spark_submit": "spark-submit --master 'local[*]'",
}

COLLECTOR_FILE = "migration_metrics_collector.py"  # relative to this notebook

# Lists every data file under each table location. If that is too slow for very large tables,
# turn it off: row counts, partitions and schema are still compared.
COLLECT_STORAGE_METRICS = True
PARALLELISM = 8  # tables collected concurrently, on each side
REPORT_MAX_PARTITION_ROWS = 500  # cap for the HTML report only; partition_diffs.csv has every row

assert "CHANGE_ME" not in OUTPUT_DIR, "Set OUTPUT_DIR (or the MIGRATION_VALIDATION_DIR env var)"
assert all("CHANGE_ME" not in p["source_db"] + p["dest_db"] for p in DATABASE_PAIRS), "Set DATABASE_PAIRS"
assert set(SIDES_TO_COLLECT) <= {"source", "dest"}, f"Unknown side in {SIDES_TO_COLLECT}"
for _side in (SOURCE, DEST):
    assert _side["collect_on"] in ("edge", "local"), f"collect_on must be 'edge' or 'local': {_side}"

---
## Helpers

Nothing in this section touches the tables or the edge node. Run it, then move on.

In [ ]:
# ── Spark session, collector module, storage I/O ────────────────────────────────────
import html
import importlib
import json
import re
import sys
from datetime import datetime, timezone

import pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession

try:
    _ = spark
    print(f"Using existing Spark session (version {spark.version})")
except NameError:
    spark = SparkSession.builder.appName("migration-validation-metrics").enableHiveSupport().getOrCreate()
    print(f"Created Spark session (version {spark.version})")

COLLECTOR_PATH = os.path.abspath(COLLECTOR_FILE)
assert os.path.isfile(COLLECTOR_PATH), (
    f"{COLLECTOR_PATH} not found: upload migration_metrics_collector.py next to this notebook"
)
sys.path.insert(0, os.path.dirname(COLLECTOR_PATH))
import migration_metrics_collector as mmc  # noqa: E402

mmc = importlib.reload(mmc)

OUTPUT_ROOT = re.sub(r"^s3n?://", "s3a://", OUTPUT_DIR.rstrip("/"))
if "://" not in OUTPUT_ROOT:
    OUTPUT_ROOT = "file://" + os.path.abspath(OUTPUT_ROOT)
RUN_DIR = f"{OUTPUT_ROOT}/{RUN_LABEL}"


def snapshot_path(side):
    return f"{RUN_DIR}/{side}_metrics.json"


def _hpath(path):
    return spark._jvm.org.apache.hadoop.fs.Path(path)


def _fs(hpath):
    return hpath.getFileSystem(spark._jsc.hadoopConfiguration())


# Snapshots and reports go through the Hadoop FileSystem on the driver, so s3a:// and local
# paths work the same way and no executor ever needs to see a local path.
def write_text(path, text):
    hp = _hpath(path)
    out = _fs(hp).create(hp, True)
    try:
        out.write(bytearray(text.encode("utf-8")))
    finally:
        out.close()


def read_text(path):
    hp = _hpath(path)
    fs = _fs(hp)
    if not fs.exists(hp):
        return None
    buf = spark._jvm.java.io.ByteArrayOutputStream()
    stream = fs.open(hp)
    try:
        spark._jvm.org.apache.hadoop.io.IOUtils.copyBytes(stream, buf, 1 << 16, False)
    finally:
        stream.close()
    return buf.toString("UTF-8")


def collector_config(side):
    cfg = SOURCE if side == "source" else DEST
    return {
        "side": side,
        "label": cfg["label"],
        "catalog": cfg["catalog"],
        "run_label": RUN_LABEL,
        "database_pairs": DATABASE_PAIRS,
        "collect_storage_metrics": COLLECT_STORAGE_METRICS,
        "parallelism": PARALLELISM,
    }


print(f"Collector: {COLLECTOR_PATH}")
print(f"Snapshots and reports: {RUN_DIR}")

In [ ]:
# ── Edge node over SSH ──────────────────────────────────────────────────────────────
import contextlib
import getpass
import posixpath
import shlex
import subprocess

try:
    import paramiko
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "paramiko"])
    import paramiko

_edge_password = None


def ssh_connect():
    global _edge_password
    assert EDGE["host"] and EDGE["user"], "Set EDGE host and user (EDGE_SSH_HOST / EDGE_SSH_USER)"
    auth = {}
    if EDGE["key_file"]:
        auth["key_filename"] = os.path.expanduser(EDGE["key_file"])
    else:
        if _edge_password is None:
            _edge_password = os.environ.get("EDGE_SSH_PASSWORD") or getpass.getpass(
                f"SSH password for {EDGE['user']}@{EDGE['host']}: "
            )
        auth["password"] = _edge_password
    client = paramiko.SSHClient()
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    client.connect(EDGE["host"], port=EDGE["port"], username=EDGE["user"], timeout=30,
                   allow_agent=False, look_for_keys=False, **auth)
    client.get_transport().set_keepalive(30)
    return client


def login_shell(cmd):
    # Same wrapping as the migration DAG: the subshell keeps `set -e` out of any EXIT trap the
    # login profile installs (e.g. `trap clear EXIT`, which fails without a TERM).
    cmd = f"(\n{cmd}\n)"
    if EDGE["cluster_type"].upper() == "MAPR":
        return f"source ~/.profile 2>/dev/null || true\n{cmd}"
    return f"bash -l <<'__LOGIN_SHELL_EOF__'\n{cmd}\n__LOGIN_SHELL_EOF__\n"


def run_remote(client, cmd, stream=False):
    """Run cmd in a login shell; returns stdout lines. Exit codes are reported via markers,
    because a noisy login profile can change the session's own exit status."""
    _, stdout, stderr = client.exec_command(login_shell(cmd))
    lines = []
    for line in iter(stdout.readline, ""):
        line = line.rstrip("\n")
        lines.append(line)
        if stream:
            print(line, flush=True)
    stderr.read()
    return lines


def marker(lines, name):
    values = [line.split("=", 1)[1] for line in lines if line.startswith(name + "=")]
    return values[-1] if values else None


SPARK_LOG_LINE = r"^[0-9][0-9/: .,-]* (INFO|WARN|DEBUG|TRACE) "

AUTH_CHECKS = {
    "kinit": 'klist -s && echo "AUTH_OK=$(klist | sed -n "s/^Default principal: //p")" || echo "AUTH_FAIL=no valid Kerberos ticket, run kinit"',
    "mapr": 'maprlogin print >/dev/null 2>&1 && echo "AUTH_OK=$(id -un)" || echo "AUTH_FAIL=no valid MapR ticket, run maprlogin"',
    "none": 'echo "AUTH_OK=not checked"',
}


def collect_on_edge(side):
    client = ssh_connect()
    try:
        lines = run_remote(client, "\n".join([
            f'mkdir -p "{EDGE["work_dir"]}" && cd "{EDGE["work_dir"]}" && echo "WORKDIR=$(pwd)"',
            AUTH_CHECKS[EDGE["auth"]],
        ]))
        work_dir = marker(lines, "WORKDIR")
        if not work_dir:
            raise RuntimeError("Could not create the edge work dir:\n" + "\n".join(lines[-20:]))
        if marker(lines, "AUTH_OK") is None:
            raise RuntimeError(f"Edge node auth check failed: {marker(lines, 'AUTH_FAIL') or lines[-5:]}")
        print(f"[{side}] edge {EDGE['user']}@{EDGE['host']}, auth: {marker(lines, 'AUTH_OK')}, dir: {work_dir}")

        remote = {name: posixpath.join(work_dir, name) for name in (
            "migration_metrics_collector.py", f"{side}_config.json", f"{side}_metrics.json", f"{side}_spark.log")}
        sftp = client.open_sftp()
        try:
            sftp.put(COLLECTOR_PATH, remote["migration_metrics_collector.py"])
            with sftp.file(remote[f"{side}_config.json"], "w") as f:
                f.write(json.dumps(collector_config(side)))
            with contextlib.suppress(OSError):
                sftp.remove(remote[f"{side}_metrics.json"])

            q = {k: shlex.quote(v) for k, v in remote.items()}
            submit = (f"{EDGE['spark_submit']} {q['migration_metrics_collector.py']} "
                      f"--config {q[f'{side}_config.json']} --output {q[f'{side}_metrics.json']}")
            # Some Spark builds log to stdout: keep everything in the log file, show all but routine log lines.
            lines = run_remote(client, "\n".join([
                f"cd {shlex.quote(work_dir)}",
                f'{{ {submit}; echo "COLLECTOR_EXIT=$?"; }} 2>&1 | tee {q[f"{side}_spark.log"]} '
                f"| grep -Ev {shlex.quote(SPARK_LOG_LINE)}",
            ]), stream=True)
            if marker(lines, "COLLECTOR_EXIT") != "0":
                raise RuntimeError(f"Collector failed on the edge node; full output: {remote[f'{side}_spark.log']}")
            with sftp.file(remote[f"{side}_metrics.json"], "r") as f:
                return f.read().decode("utf-8")
        finally:
            sftp.close()
    finally:
        client.close()

---
## Step 1: Collect metrics

Collects each side in `SIDES_TO_COLLECT` and saves it to `<side>_metrics.json` under
`OUTPUT_DIR/RUN_LABEL`, overwriting that side's previous snapshot. On the edge node, progress
lines are streamed here as tables finish; the complete output, including Spark's own log, stays in
`<side>_spark.log` in the remote work dir.

In [ ]:
for _side in SIDES_TO_COLLECT:
    if (SOURCE if _side == "source" else DEST)["collect_on"] == "edge":
        _text = collect_on_edge(_side)
    else:
        _text = json.dumps(mmc.collect_side(spark, collector_config(_side), emit=print))
    write_text(snapshot_path(_side), _text)
    print(f"[{_side}] snapshot saved: {snapshot_path(_side)}\n")

---
## Step 2: Compare

Reads both snapshots back from `OUTPUT_DIR` and compares them table by table. Tables are
matched by name within each database pair.

In [ ]:
_texts = {side: read_text(snapshot_path(side)) for side in ("source", "dest")}
missing = [side for side, text in _texts.items() if text is None]
summary_df = partitions_df = None
if missing:
    print(f"Comparison skipped: no snapshot yet for {missing} under {RUN_DIR}. Collect it first.")
else:
    src_snapshot, dest_snapshot = json.loads(_texts["source"]), json.loads(_texts["dest"])
    for snap in (src_snapshot, dest_snapshot):
        print(f"{snap['side']:>6}: {snap['label']} collected {snap['collected_at']} "
              f"(Spark {snap['spark_version']}, Python {snap.get('python_version', '?')})")
    _rows, _diffs, _warnings = mmc.compare_snapshots(src_snapshot, dest_snapshot)
    for w in _warnings:
        print(f"WARNING: {w}")

    summary_cols = [
        "pair", "table", "status", "issues", "source_rows", "dest_rows", "source_partitions", "dest_partitions",
        "source_files", "dest_files", "source_bytes", "dest_bytes", "partition_issues", "schema_differences",
        "partition_schema_differences", "notes", "source_location", "dest_location",
    ]
    diff_cols = ["pair", "table", "partition", "issue", "source_rows", "dest_rows",
                 "source_files", "dest_files", "source_bytes", "dest_bytes"]
    summary_df = pd.DataFrame(_rows).reindex(columns=summary_cols)
    partitions_df = pd.DataFrame(_diffs).reindex(columns=diff_cols)
    for _df, _cols in ((summary_df, summary_cols[4:13]), (partitions_df, diff_cols[4:])):
        for _col in _cols:
            _df[_col] = _df[_col].astype("Int64")

    print()
    print(summary_df["status"].value_counts().to_string())
    display(summary_df)

---
## Step 3: Write the report

Writes `summary.csv`, `partition_diffs.csv` and `report.html` to a new `report_<UTC timestamp>/` folder.

In [ ]:
STATUS_COLORS = {
    "PASS": "#d4edda", "WARN": "#fff3cd", "FAIL": "#f8d7da", "ERROR": "#f8d7da",
    "MISSING_ON_DEST": "#f8d7da", "EXTRA_ON_DEST": "#e2e3e5", "SKIPPED_VIEW": "#e2e3e5",
}


def _html_table(df):
    shown = df.copy()
    for col in [c for c in shown.columns if c.endswith("_bytes")]:
        shown[col] = shown[col].map(lambda n: "" if pd.isna(n) else mmc.fmt_bytes(int(n)))
    out = shown.to_html(index=False, na_rep="", escape=True, border=0)
    for status, color in STATUS_COLORS.items():
        out = out.replace(f"<td>{status}</td>", f'<td style="background:{color};font-weight:bold">{status}</td>')
    return out


def render_report(summary, diffs, src_snap, dest_snap):
    counts = " &nbsp; ".join(f"<b>{html.escape(k)}</b>: {v}" for k, v in summary["status"].value_counts().items())
    shown_note = ""
    if len(diffs) > REPORT_MAX_PARTITION_ROWS:
        shown_note = f", first {REPORT_MAX_PARTITION_ROWS} shown, see partition_diffs.csv"
    return f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>Migration validation: {html.escape(RUN_LABEL)}</title>
<style>
 body {{ font-family: sans-serif; font-size: 13px; margin: 24px; }}
 table {{ border-collapse: collapse; margin-bottom: 24px; }}
 th, td {{ border: 1px solid #ccc; padding: 4px 8px; vertical-align: top; }}
 th {{ background: #f2f2f2; }}
</style></head><body>
<h1>Migration validation: {html.escape(src_snap['label'])} &rarr; {html.escape(dest_snap['label'])}</h1>
<p>Run label <b>{html.escape(RUN_LABEL)}</b>.
 Source collected {html.escape(src_snap['collected_at'])}, dest collected {html.escape(dest_snap['collected_at'])}.</p>
<p>{counts}</p>
<h2>Tables</h2>
{_html_table(summary)}
<h2>Partition differences ({len(diffs):,}{shown_note})</h2>
{_html_table(diffs.head(REPORT_MAX_PARTITION_ROWS)) if len(diffs) else "<p>None.</p>"}
</body></html>"""


if summary_df is None:
    print("No comparison to report; see Step 2.")
else:
    report_dir = f"{RUN_DIR}/report_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
    write_text(f"{report_dir}/summary.csv", summary_df.to_csv(index=False))
    write_text(f"{report_dir}/partition_diffs.csv", partitions_df.to_csv(index=False))
    write_text(f"{report_dir}/report.html", render_report(summary_df, partitions_df, src_snapshot, dest_snapshot))
    print(f"Report written to {report_dir}/")

---
## Drill-down: one table

Lists every partition-level difference for one table: rows per partition, file count and bytes
per partition directory, and partition directories that are not registered.

In [ ]:
TABLE_TO_INSPECT = ""  # table name, e.g. "orders"

if TABLE_TO_INSPECT and summary_df is not None:
    _mask = summary_df["table"].str.lower() == TABLE_TO_INSPECT.lower()
    display(summary_df[_mask].T)
    display(partitions_df[partitions_df["table"].str.lower() == TABLE_TO_INSPECT.lower()])